# Trích feature từ encoder Shift-GCN (đóng băng) của Neuron

Code lấy từ `zeroshot/nesy` trong repo `thesis`. Đây là bản Neuron đã sửa: phép Shift chạy bằng PyTorch thuần nên **không cần build CUDA**.

1. Kiểm tra bản PyTorch thuần cho ra đúng feature như kernel CUDA, so với file đã xuất trước đó.
2. Xuất feature đã pool `(N, 256)` cho 4 split vào `neuron_extract/feats/<task>.npz`, gồm các key `train_*`, `zsl_*`, `gzsl_*`. Phase 1 dùng các file này.

Đã kiểm tra ngày 2026-11-07: `fc` của encoder đạt 89.7% (55/5) và 89.1% (48/12) trên mẫu test thuộc lớp seen.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, subprocess
DRIVE_DIR   = '/content/drive/MyDrive/BaoHuy/zeroshot'
WEIGHTS_DIR = f'{DRIVE_DIR}/shiftgcn_weights'   # 12 file .pt của Neuron (pretrain trên lớp seen)
SKEL = {
    'ntu60':  '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz',
    'ntu120': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU120_CSub.npz',
}
OUT_DIR  = f'{DRIVE_DIR}/neuron_extract'
FEAT_OUT = f'{OUT_DIR}/feats'
os.makedirs(FEAT_OUT, exist_ok=True)

# Lớp unseen của từng split (chỉ số 0-based), giống nesy/feeders/feeder_ntu.py
UNSEEN = {
    'ntu60_xsub_seen55_unseen5':    [10, 11, 19, 26, 56],
    'ntu60_xsub_seen48_unseen12':   [3, 5, 9, 12, 15, 40, 42, 47, 51, 56, 58, 59],
    'ntu120_xsub_seen110_unseen10': [4, 13, 37, 43, 49, 65, 88, 95, 99, 106],
    'ntu120_xsub_seen96_unseen24':  [5, 9, 11, 16, 18, 20, 22, 29, 35, 39, 45, 49, 59, 68, 70, 81, 84, 87, 93, 94, 104, 113, 114, 119],
}
TASKS = list(UNSEEN)
print('weights:', len(glob.glob(f'{WEIGHTS_DIR}/**/*.pt', recursive=True)), 'file')
print('skeleton:', {k: os.path.exists(v) for k, v in SKEL.items()})

In [ ]:
# Repo private thì thêm token GitHub vào Colab > Secrets với tên GH_TOKEN (quyền đọc repo là đủ).
REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)   # subprocess để không in token ra output
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
NESY = f'{REPO}/zeroshot/nesy'
!git -C {REPO} log -1 --format='%h %s'

In [ ]:
import numpy as np, pandas as pd, torch, time
os.chdir(NESY)
if NESY not in sys.path:
    sys.path.insert(0, NESY)
for k in [k for k in sys.modules if k.split('.')[0] in ('model', 'graph', 'feeders')]:
    del sys.modules[k]   # chạy lại cell thì nạp lại code mới pull
from model.shiftgcn_match import Model
from model.Temporal_shift.cuda import shift as shift_mod
from feeders import tools
print('Shift backend:', shift_mod.BACKEND)

def weight_path(task):
    ds, prot, s, u = task.split('_')
    hits = glob.glob(f'{WEIGHTS_DIR}/**/{ds}_ShiftGCN_joint_{prot}_{s}_{u}-*.pt', recursive=True)
    assert len(hits) == 1, (task, hits)
    return hits[0]

def load_encoder(task):
    sd = torch.load(weight_path(task), map_location='cpu')
    sd = {k.split('module.')[-1]: v for k, v in sd.items()}
    m = Model(num_class=sd['fc.weight'].shape[0], num_point=25, num_person=2,
              graph='graph.ntu_rgb_d.Graph', graph_args={'labeling_mode': 'spatial'})
    m.load_state_dict(sd, strict=True)   # strict: báo lỗi nếu thiếu hoặc thừa key
    return m.cuda().eval()

def preprocess(x, p_interval=(0.95,), window=64):
    # x: (T, 150) định dạng CTR-GCN -> (C, T, V, M), giống Feeder_Shiftgcn_Match.__getitem__ (không augment).
    # p_interval=[0.95]: center crop 95%, đúng như lúc test khi pretrain Shift-GCN.
    T = x.shape[0]
    d = x.reshape(T, 2, 25, 3).transpose(3, 0, 2, 1)
    valid = int(np.sum(d.sum(0).sum(-1).sum(-1) != 0))
    return tools.valid_crop_resize(d, valid, list(p_interval), window)

@torch.no_grad()
def extract(model, X, bs=256, keep_cube=False):
    pooled, cubes = [], []
    for i in range(0, len(X), bs):
        xb = np.stack([preprocess(x) for x in X[i:i + bs]])
        _, f, cube = model(torch.from_numpy(xb).float().cuda())  # f: (n, 256); cube: (n, 256, 16, 25), trung bình qua 2 người
        pooled.append(f.cpu())
        if keep_cube:
            cubes.append(cube.half().cpu())
    pooled = torch.cat(pooled).numpy()
    return (pooled, torch.cat(cubes).numpy()) if keep_cube else pooled

def extract_idx(model, X, idx, chunk=4096):
    # lấy theo từng khúc chỉ số để không phải copy cả x_train (7-11 GB) ra bộ nhớ
    return np.concatenate([extract(model, X[idx[i:i + chunk]]) for i in range(0, len(idx), chunk)])

def cos_rows(a, b):
    a = a / np.linalg.norm(a, axis=1, keepdims=True).clip(1e-8)
    b = b / np.linalg.norm(b, axis=1, keepdims=True).clip(1e-8)
    return (a * b).sum(1)

enc = load_encoder(TASKS[0])
print('encoder OK, fc out =', enc.fc.out_features)

In [ ]:
# Shift bản PyTorch thuần phải cho ra cùng feature với kernel CUDA.
# feats/ntu60_xsub_seen55_unseen5.npz đã được xuất bằng kernel CUDA (2026-11-07), dùng làm chuẩn so sánh.
# Kỳ vọng: cos min > 0.9999, max |diff| ~1e-5 (chỉ sai số float).
task = TASKS[0]
ref_path = f'{FEAT_OUT}/{task}.npz'
if os.path.exists(ref_path):
    ref = np.load(ref_path)['gzsl_data'][:2048]
    X_test = np.load(SKEL['ntu60'])['x_test']
    t0 = time.time()
    f = extract(load_encoder(task), X_test[:2048])
    print(f'{time.time() - t0:.0f}s  max |diff| = {np.abs(f - ref).max():.2e}  '
          f'cos min = {cos_rows(f, ref).min():.6f}  cos mean = {cos_rows(f, ref).mean():.6f}')
else:
    print('chưa có file tham chiếu, bỏ qua bước kiểm tra')

In [ ]:
# Xuất feature đã pool cho từng split. Split nào đã có file thì chỉ đọc lại.
# NCM (cosine tới tâm lớp) để kiểm tra nhanh chất lượng, không cần train gì:
#   seen_ncm          : tâm lớp lấy từ train -> phân loại mẫu test thuộc lớp seen
#   unseen_oracle_ncm : tâm lớp lấy ngay từ mẫu test unseen (oracle) = trần cho zero-shot trên feature này
def ncm_acc(tr_f, tr_y, te_f, te_y):
    cls = np.unique(te_y)
    mu = np.stack([tr_f[tr_y == c].mean(0) for c in cls])
    mu /= np.linalg.norm(mu, axis=1, keepdims=True)
    te = te_f / np.linalg.norm(te_f, axis=1, keepdims=True)
    return float((cls[(te @ mu.T).argmax(1)] == te_y).mean())

rows, loaded_ds = [], None
for task in TASKS:
    ds = task.split('_')[0]
    path = f'{FEAT_OUT}/{task}.npz'
    if os.path.exists(path):
        out = dict(np.load(path))
        print(task, '| đã có, đọc lại')
    else:
        if loaded_ds != ds:   # mỗi dataset chỉ đọc skeleton một lần
            X_train = X_test = skel = None
            skel = np.load(SKEL[ds])
            X_train, y_train = skel['x_train'], np.argmax(skel['y_train'], 1)
            X_test, y_test = skel['x_test'], np.argmax(skel['y_test'], 1)
            loaded_ds = ds
        tr_idx = np.where(~np.isin(y_train, UNSEEN[task]))[0]
        zmask = np.isin(y_test, UNSEEN[task])
        t0 = time.time()
        enc = load_encoder(task)
        ftr = extract_idx(enc, X_train, tr_idx)
        fte = extract(enc, X_test)
        out = dict(train_data=ftr, train_label=y_train[tr_idx],
                   zsl_data=fte[zmask], zsl_label=y_test[zmask],
                   gzsl_data=fte, gzsl_label=y_test)
        np.savez(path, **out)
        print(task, f'{time.time() - t0:.0f}s')
    assert set(np.unique(out['zsl_label'])) == set(UNSEEN[task])
    assert not set(np.unique(out['train_label'])) & set(UNSEEN[task])
    smask = ~np.isin(out['gzsl_label'], UNSEEN[task])
    rows.append(dict(task=task, n_train=len(out['train_label']), n_zsl=len(out['zsl_label']), n_gzsl=len(out['gzsl_label']),
                     seen_ncm=ncm_acc(out['train_data'], out['train_label'], out['gzsl_data'][smask], out['gzsl_label'][smask]),
                     unseen_oracle_ncm=ncm_acc(out['zsl_data'], out['zsl_label'], out['zsl_data'], out['zsl_label'])))

feat_q = pd.DataFrame(rows).round(4)
feat_q.to_csv(f'{OUT_DIR}/feature_quality.csv', index=False)
display(feat_q)
!ls -lh {FEAT_OUT}

## Đọc kết quả

- **`verify-shift`**: cần `cos min > 0.9999`. Nếu thấp hơn thì bản Shift PyTorch thuần có lỗi. Khi đó đặt `os.environ['SHIFT_BACKEND'] = 'cuda'` và build kernel như trước để dùng tạm.
- **`export`**: NTU60 đã có số tham chiếu, `seen_ncm` 0.88, `unseen_oracle_ncm` 0.88 (55/5) và 0.92 (48/12). NTU120 thì đây là lần đầu có số.

Gửi lại output của hai cell này.